# 预习 3：NLP / CV / LLM 总览与 Transformer 前置概念

> 建议学习顺序：预习 1 → 预习 2 → 预习 3 → 原 Day1 Attention。  
> 这份 notebook 的目标是把 NLP、CV、大模型、Transformer 之前的概念串起来，让你知道后面 7 天为什么重点学 Attention、KV Cache、RoPE、PagedAttention、MoE、LoRA、RAG、Agent。

---

## 这份 notebook 解决什么问题？

你现在要学的是大模型核心技术，但大模型不是孤立知识点。  
它连接了 NLP、CV、表示学习、生成模型、检索系统和推理部署。

本 notebook 会按下面顺序讲：

1. NLP 主流方法演进
2. 文本如何变成模型能处理的数字
3. CV 主流方法演进
4. Transformer 为什么统一了 NLP / CV / 多模态
5. LLM 是怎么训练、对齐、部署、应用的
6. 为什么你后面 7 天的主题顺序是合理的

# 0. 环境准备

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

np.random.seed(42)
print("ready")

ready


# 1. NLP 发展脉络：从规则到大语言模型

NLP，即自然语言处理，目标是让机器理解和生成自然语言。

## 1.1 规则系统时代

早期 NLP 很依赖人工规则：

- 词典
- 正则
- 语法规则
- 专家系统

优点：可解释。  
缺点：泛化差，维护成本极高。

## 1.2 统计 NLP

代表方法：

- n-gram 语言模型
- HMM
- CRF
- TF-IDF
- 朴素贝叶斯

核心思想：用统计规律处理文本。

## 1.3 深度学习 NLP

代表方法：

- Word2Vec / GloVe
- RNN / LSTM / GRU
- Seq2Seq
- Attention

## 1.4 Transformer / Pretraining 时代

代表模型：

- BERT：encoder-only，擅长理解
- GPT：decoder-only，擅长生成
- T5：encoder-decoder，文本到文本
- LLaMA / Qwen / DeepSeek / Claude / GPT：现代 LLM 家族

核心变化：

> 从针对具体任务训练模型，变成先大规模预训练，再通过指令微调、对齐、RAG、工具调用适配任务。

# 2. 文本为什么不能直接喂给模型？

模型只能处理数字。  
所以文本需要经过：

```text
原始文本 -> 分词 tokenization -> token id -> embedding 向量 -> 模型
```

例如：

```text
"我喜欢机器学习"
```

可能被切成：

```text
["我", "喜欢", "机器", "学习"]
```

也可能被切成：

```text
["我", "喜欢", "机器学习"]
```

也可能用 BPE / SentencePiece 切成子词片段。

## 为什么分词重要？

分词会影响：

- 序列长度
- 稀有词处理
- 多语言能力
- 推理速度
- context window 实际可容纳内容

In [2]:
text = "我 喜欢 机器 学习 ， 也 喜欢 深度 学习"
tokens = text.split()
vocab = {tok: idx for idx, tok in enumerate(sorted(set(tokens)))}
ids = [vocab[tok] for tok in tokens]

print("tokens:", tokens)
print("vocab:", vocab)
print("token ids:", ids)

tokens: ['我', '喜欢', '机器', '学习', '，', '也', '喜欢', '深度', '学习']
vocab: {'也': 0, '喜欢': 1, '学习': 2, '我': 3, '机器': 4, '深度': 5, '，': 6}
token ids: [3, 1, 4, 2, 6, 0, 1, 5, 2]


# 3. Bag-of-Words：最经典的文本向量化

Bag-of-Words 不关心顺序，只统计词频。

例如：

```text
句子 A：我 喜欢 机器 学习
句子 B：我 喜欢 深度 学习
```

如果词表是：

```text
[我, 喜欢, 机器, 深度, 学习]
```

那么句子可以变成词频向量：

```text
A = [1, 1, 1, 0, 1]
B = [1, 1, 0, 1, 1]
```

缺点：

- 不懂语义
- 不懂顺序
- 维度很高
- 稀疏

In [3]:
docs = [
    "我 喜欢 机器 学习",
    "我 喜欢 深度 学习",
    "深度 学习 改变 自然语言处理",
    "机器 学习 包括 监督 学习 和 无监督 学习",
]

tokenized = [doc.split() for doc in docs]
vocab = sorted(set(tok for doc in tokenized for tok in doc))
word2idx = {w: i for i, w in enumerate(vocab)}

bow = np.zeros((len(docs), len(vocab)), dtype=int)
for i, doc in enumerate(tokenized):
    for tok in doc:
        bow[i, word2idx[tok]] += 1

print("vocab:", vocab)
print("BoW matrix shape:", bow.shape)
print(bow)

vocab: ['包括', '和', '喜欢', '学习', '我', '改变', '无监督', '机器', '深度', '监督', '自然语言处理']
BoW matrix shape: (4, 11)
[[0 0 1 1 1 0 0 1 0 0 0]
 [0 0 1 1 1 0 0 0 1 0 0]
 [0 0 0 1 0 1 0 0 1 0 1]
 [1 1 0 3 0 0 1 1 0 1 0]]


# 4. TF-IDF：不仅看出现次数，还看区分度

TF-IDF 的思想：

- 一个词在当前文档出现多，说明它重要
- 但如果它在所有文档都常见，区分度就低

所以：

$$
\text{TF-IDF}(t,d)=\text{TF}(t,d)\cdot \text{IDF}(t)
$$

其中 IDF 通常类似：

$$
\text{IDF}(t)=\log\frac{N}{df(t)}
$$

TF-IDF 曾经是搜索、文本分类、信息检索里的核心特征。

In [4]:
N = len(docs)
df = np.count_nonzero(bow > 0, axis=0)
idf = np.log((N + 1) / (df + 1)) + 1
tfidf = bow * idf

np.set_printoptions(precision=2, suppress=True)
print("idf:", dict(zip(vocab, idf)))
print("\nTF-IDF matrix:")
print(tfidf)

idf: {'包括': np.float64(1.916290731874155), '和': np.float64(1.916290731874155), '喜欢': np.float64(1.5108256237659907), '学习': np.float64(1.0), '我': np.float64(1.5108256237659907), '改变': np.float64(1.916290731874155), '无监督': np.float64(1.916290731874155), '机器': np.float64(1.5108256237659907), '深度': np.float64(1.5108256237659907), '监督': np.float64(1.916290731874155), '自然语言处理': np.float64(1.916290731874155)}

TF-IDF matrix:
[[0.   0.   1.51 1.   1.51 0.   0.   1.51 0.   0.   0.  ]
 [0.   0.   1.51 1.   1.51 0.   0.   0.   1.51 0.   0.  ]
 [0.   0.   0.   1.   0.   1.92 0.   0.   1.51 0.   1.92]
 [1.92 1.92 0.   3.   0.   0.   1.92 1.51 0.   1.92 0.  ]]


# 5. Embedding：从稀疏词频到稠密语义向量

Embedding 把 token id 映射成连续向量：

$$
E[token\_id] \rightarrow \mathbb{R}^d
$$

例如：

```text
"猫" -> [0.12, -0.33, 0.08, ...]
"狗" -> [0.10, -0.29, 0.11, ...]
```

如果训练得好，语义相近的词在向量空间里也更接近。

## LLM 中的 embedding

在 Transformer 里，第一步就是把 token id 转成 embedding：

$$
X = \text{Embedding}(input\_ids)
$$

后面的 Attention、MLP、LayerNorm 都是在这些向量上操作。

In [5]:
# 一个 toy embedding lookup
vocab_size = len(vocab)
d_model = 6
embedding_table = np.random.randn(vocab_size, d_model) * 0.1

sentence = "我 喜欢 深度 学习".split()
ids = [word2idx[t] for t in sentence]
X = embedding_table[ids]

print("sentence:", sentence)
print("ids:", ids)
print("embedding shape:", X.shape)
print(X)

sentence: ['我', '喜欢', '深度', '学习']
ids: [4, 2, 8, 3]
embedding shape: (4, 6)
[[-0.05  0.01 -0.12  0.04 -0.06 -0.03]
 [ 0.02 -0.19 -0.17 -0.06 -0.1   0.03]
 [ 0.03 -0.18  0.03 -0.04 -0.07  0.06]
 [-0.09 -0.14  0.15 -0.02  0.01 -0.14]]


# 6. 语言模型：预测下一个 token

语言模型的目标：

$$
P(x_1, x_2, ..., x_T)
$$

根据链式法则：

$$
P(x_1, ..., x_T)=\prod_{t=1}^{T}P(x_t \mid x_{<t})
$$

自回归语言模型训练的是：

> 给定前面的 token，预测下一个 token。

例如：

```text
输入：我 喜欢 机器
目标：学习
```

这就是你后面 Day4 自回归生成的基础。

# 7. Seq2Seq 与 Attention：Transformer 之前的重要桥梁

在 Transformer 之前，机器翻译常用 Seq2Seq：

```text
Encoder RNN 读完整句 -> 压成一个向量 -> Decoder RNN 逐词生成
```

问题：

> 整个输入句子被压缩进一个固定长度向量，长句信息容易丢失。

Attention 的出现解决了这个问题：

> Decoder 每生成一个 token，都可以动态查看输入序列的不同位置。

这就是 Attention 的历史动机。  
Transformer 则进一步把 Attention 变成整个模型的核心结构。

# 8. CV 发展脉络：从手工特征到视觉大模型

CV，即计算机视觉，目标是让机器理解图像和视频。

## 8.1 传统视觉

代表方法：

- SIFT
- HOG
- Haar features
- 边缘检测
- 光流

特点：大量人工设计特征。

## 8.2 CNN 时代

代表模型：

- LeNet
- AlexNet
- VGG
- ResNet
- EfficientNet
- YOLO / Faster R-CNN / Mask R-CNN

CNN 的优势是利用图像局部结构。

## 8.3 Transformer / 多模态时代

代表方向：

- ViT：把图像切成 patch，当作 token 序列
- CLIP：图文对齐
- Diffusion：文本生成图像
- 多模态 LLM：文本、图像、语音统一建模

# 9. ViT 的核心：把图像切成 patch token

Vision Transformer 的基本思路：

```text
图像 -> 切成 patches -> 每个 patch 展平 -> Linear 投影 -> token 序列 -> Transformer
```

例如一张 $32 \times 32$ 图像，patch size 为 $8 \times 8$：

```text
patch 数量 = (32/8) * (32/8) = 16
```

这样图像就变成了长度为 16 的 token 序列。

In [ ]:
# patchify 一个 toy image
image = np.arange(32 * 32).reshape(32, 32)
patch_size = 8

patches = []
for i in range(0, image.shape[0], patch_size):
    for j in range(0, image.shape[1], patch_size):
        patch = image[i:i+patch_size, j:j+patch_size]
        patches.append(patch.flatten())

patches = np.array(patches)
print("image shape:", image.shape)
print("patches shape:", patches.shape)  # num_patches, patch_dim

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title("Toy Image")
plt.axis("off")
plt.show()

# 10. LLM 是什么？

LLM，即 Large Language Model，大语言模型。  
它通常具有以下特征：

1. 基于 Transformer，尤其是 decoder-only Transformer
2. 在海量文本或多模态数据上预训练
3. 参数量大，从数十亿到数千亿不等
4. 具备上下文学习、指令跟随、推理、代码生成等能力
5. 可以通过 RAG、工具调用、Agent 框架接入外部知识和行动能力

## 但要注意

LLM 不是“真正理解一切”的魔法系统。  
它本质上仍然是在上下文条件下生成下一个 token，只是训练规模和表示能力让它表现出复杂能力。

# 11. LLM 训练流程总览

现代 LLM 通常经历：

## 11.1 预训练 Pretraining

目标：学习语言和世界知识的统计规律。

常见任务：

```text
预测下一个 token
```

## 11.2 指令微调 Instruction Tuning / SFT

让模型学会按照人类指令回答。

数据形式：

```text
Instruction -> Response
```

## 11.3 偏好对齐 Alignment

让模型回答更符合人类偏好和安全要求。

常见方法：

- RLHF
- RLAIF
- DPO
- PPO 类方法

## 11.4 工具增强 / RAG / Agent

让模型不只依赖参数记忆，还能：

- 检索资料
- 调用工具
- 写代码
- 查数据库
- 分步骤执行任务

# 12. LLM 模型类型：Encoder-only / Decoder-only / Encoder-Decoder

## 12.1 Encoder-only

代表：BERT

适合：

- 文本分类
- 检索 embedding
- 句子理解
- NER

特点：双向看上下文，不直接自回归生成长文本。

## 12.2 Decoder-only

代表：GPT、LLaMA、Qwen、DeepSeek 等

适合：

- 对话
- 文本生成
- 代码生成
- Agent

特点：从左到右预测下一个 token。

## 12.3 Encoder-Decoder

代表：T5、BART

适合：

- 翻译
- 摘要
- 文本改写
- 输入到输出的转换任务

# 13. 为什么现代 LLM 多用 Decoder-only？

因为自回归生成非常统一：

```text
任何任务都可以包装成 prompt -> completion
```

例如：

```text
翻译：把下面英文翻译成中文：...
分类：判断下面评论是正面还是负面：...
代码：写一个 Python 函数：...
推理：一步一步分析：...
```

Decoder-only 模型把这些任务都变成了“继续生成文本”。

这也是你后面 Day4 学自回归生成的原因。

# 14. RAG：为什么大模型还需要检索？

LLM 参数里有很多知识，但存在几个问题：

1. 知识可能过时
2. 容易幻觉
3. 私有数据不在训练集中
4. 长尾知识记不准
5. 需要可追溯来源

RAG 的流程：

```text
用户问题 -> 检索相关文档 -> 拼入上下文 -> LLM 基于文档回答
```

核心组件：

- 文档切分 chunking
- embedding
- 向量数据库
- reranker
- prompt assembly
- answer generation
- citation / grounding

你 Day7 里的 RAG-Agent 就是这个方向。

# 15. Fine-tuning / LoRA / QLoRA：什么时候需要微调？

微调不是万能药。  
一般适合：

- 固定格式输出
- 特定风格
- 领域术语
- 任务模式稳定
- 小模型适配特定场景

不适合：

- 让模型记大量实时知识
- 频繁变化的事实
- 替代检索系统
- 修复所有幻觉

LoRA 的思想是：

> 不更新原模型全部参数，而是在部分线性层旁边加低秩增量矩阵。

这就是 Day7 的重点。

# 16. Prompt Engineering：提示词不是玄学，而是接口设计

好的 prompt 应该明确：

1. 角色 / 任务
2. 输入数据
3. 输出格式
4. 约束条件
5. 评分标准
6. 边界情况
7. 示例

对于 LLM 来说，prompt 就像临时程序。  
但 prompt 不是强约束，所以复杂任务需要：

- 结构化输出
- 工具调用
- 校验器
- 检索
- 工作流编排

# 17. Agent：LLM + 工具 + 状态 + 规划

Agent 不是“更聪明的聊天机器人”，而是一个系统结构：

```text
目标 -> 规划 -> 调用工具 -> 观察结果 -> 更新状态 -> 继续行动
```

常见组件：

- planner
- memory
- tools
- retriever
- executor
- evaluator
- guardrails

但 Agent 容易出问题：

- 幻觉工具结果
- 循环调用
- 计划漂移
- 状态污染
- 成本不可控

所以真正工程化的 Agent 通常要有强约束和可观测 trace。

# 18. 你后面 7 天笔记的学习路线为什么合理？

你现有 7 天其实是一条很清晰的大模型核心路线：

| Day | 主题 | 位置 |
|---|---|---|
| Day1 | Scaled Dot-Product Attention | Transformer 核心公式 |
| Day2 | MHA / MQA / GQA / KV Cache | Attention 工程化与显存 |
| Day3 | RoPE | 位置信息如何进入模型 |
| Day4 | Autoregressive Generation / KV Cache | LLM 生成机制 |
| Day5 | PagedAttention / Continuous Batching | 大模型 serving 系统 |
| Day6 | MoE Router / Top-K Experts | 大模型扩容架构 |
| Day7 | LoRA / QLoRA / RAG / Agent | 应用与适配 |

所以这三份预习笔记的目的不是替代原计划，而是补地基。

# 19. 进入 Day1 前你必须掌握的最小前置清单

如果下面内容基本清楚，就可以正式进入 Attention：

## 数学与张量

- 向量、矩阵、张量
- 点积
- 矩阵乘法
- softmax
- batch 维度 `[B, T, d]`

## 机器学习

- 训练集 / 验证集 / 测试集
- loss
- gradient descent
- overfitting
- generalization

## 深度学习

- Linear layer
- activation
- embedding
- MLP
- CNN/RNN/LSTM/GNN 的基本用途

## NLP / LLM

- tokenization
- token id
- embedding
- language modeling
- next-token prediction
- encoder-only / decoder-only / encoder-decoder

# 20. 自测题

1. NLP 中 tokenization 为什么重要？
2. Bag-of-Words 和 embedding 的区别是什么？
3. 为什么语言模型可以通过预测下一个 token 学到很多能力？
4. Seq2Seq 为什么需要 Attention？
5. ViT 如何把图像变成 Transformer 能处理的 token 序列？
6. Encoder-only、Decoder-only、Encoder-Decoder 分别适合什么任务？
7. 为什么 RAG 不能简单等同于“联网搜索”？
8. LoRA 适合解决什么问题？不适合解决什么问题？
9. Agent 相比普通 LLM 调用多了哪些系统组件？
10. 你后面 7 天中，哪几天偏模型结构，哪几天偏推理部署，哪几天偏应用适配？

完成后，你可以正式进入原来的 Day1：Scaled Dot-Product Attention。

# 21. 最终学习建议

## 推荐顺序

```text
预习 1：ML / DL 基础
预习 2：神经网络架构地图
预习 3：NLP / CV / LLM 总览
Day1：Scaled Dot-Product Attention
Day2：MHA / MQA / GQA / KV Cache
Day3：RoPE
Day4：Autoregressive Generation / KV Cache
Day5：PagedAttention / Continuous Batching
Day6：MoE
Day7：LoRA / QLoRA / RAG / Agent
```

## 学习策略

不要追求第一次就把所有细节吃透。  
你应该先建立结构：

1. 这个概念解决什么问题？
2. 它的输入输出是什么？
3. 它相比旧方法解决了什么痛点？
4. 它有什么代价？
5. 它在大模型链路中处于哪个位置？

只要你能按这个框架理解，后面再看公式和代码会轻松很多。